# Semana 4 — Tarea del proyecto ancla · Zonas de operación de la planta
### Cuaderno guiado (ALUMNO)

En el cuaderno satélite aprendiste a agrupar datos sin etiquetas con k-means, a elegir el número de
grupos y a proyectar datos con PCA. Ahora lo aplicas a la planta de flotación: **agrupar las horas
según cómo estaba operando la planta, sin mirar la calidad, y comprobar después si en alguna de esas
formas de operar la calidad empeora**.

**Sobre los datos.** Usamos la misma tabla horaria **semisintética** de las semanas anteriores
(`flotacion_semisintetica_horaria.csv`): las variables de proceso son las reales de la planta y la
sílice del concentrado está simulada con un modelo del proceso que no se publica. La columna
`fuera_spec` vale 1 cuando la sílice de esa hora supera el 3 %.

**Qué te damos hecho.** La sección 1 está resuelta: carga la tabla, fija qué variables entran en el
agrupamiento y las estandariza. Solo tienes que ejecutarla y leerla.

**Qué entregas esta semana (tarea semanal)**

- Las **secciones 2 a 5**: un **k-means con 5 grupos** sobre las variables de proceso, el **perfil de
  calidad** de cada grupo, la comprobación, con el **codo y la silueta**, de si esos grupos son nítidos
  y una vista de las zonas en un plano con **PCA**. Completa las celdas `# TODO` y responde a las cuatro
  preguntas en las celdas «Tu respuesta».
- Se entrega como archivo `.ipynb` en el Campus Virtual, ejecutado de principio a fin sin errores.

**Sin parte de proyecto final.** En la Semana 4 todo el trabajo es esta tarea semanal: no hay
ampliaciones para el proyecto final.

**Cómo trabajar**

- Ejecuta las celdas **en orden**, de arriba abajo.
- En las celdas con `# TODO` hay huecos marcados con `None`. Complétalos siguiendo la **pista** del
  comentario: casi siempre es una sola línea.
- Después de cada paso importante hay una celda **Comprueba**. Si da error, revisa ese paso antes de
  seguir: los pasos siguientes dependen de él.
- Un resultado modesto **no es un fracaso**: lo importante es que el método sea correcto y que
  interpretes el resultado con honestidad.

## 0. Preparación

Cargamos las librerías. Ejecuta la celda sin cambiar nada.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

pd.set_option('display.width', 120)

## 1. Los datos (resuelto)

Esta sección ya está hecha: ejecuta sus celdas sin cambiar nada y lee los resultados. Las secciones
siguientes usan las variables que se definen aquí: `tabla`, `variables_proceso`, `escalador` y
`X_proceso`.

### 1a. Cargar la tabla

La tabla tiene **una fila por hora** (marzo–septiembre de 2017).

| Columna | Qué contiene | Papel en la tarea |
|---|---|---|
| `date` | hora a la que corresponde la fila | metadato |
| `n_muestras` | número de registros de 20 s promediados en esa hora | metadato |
| `silice` | % de sílice en el concentrado (semisintética) | **solo para describir** los grupos |
| `iron_feed`, `silica_feed` | % de hierro y de sílice del mineral que entra | entra en el agrupamiento |
| `almidon_medio` | caudal de almidón | entra en el agrupamiento |
| `amina_media` | caudal de amina | entra en el agrupamiento |
| `caudal_pulpa` | caudal de pulpa | entra en el agrupamiento |
| `ph_medio` | pH de la pulpa | entra en el agrupamiento |
| `densidad` | densidad de la pulpa | entra en el agrupamiento |
| `aire_medio` | aire inyectado, media de las 7 columnas de flotación | entra en el agrupamiento |
| `nivel_medio` | nivel de espuma, media de las 7 columnas | entra en el agrupamiento |
| `fuera_spec` | 1 si la sílice de esa hora supera el 3 %, 0 si no | **solo para describir** los grupos |

Los datos están en la carpeta `datasets`, al lado de la carpeta de esta semana. Si la celda siguiente
da `FileNotFoundError`, revisa que has descargado el CSV en `SII/datasets/`.

In [ ]:
ruta_datos = '../datasets/flotacion_semisintetica_horaria.csv'
tabla = pd.read_csv(ruta_datos, parse_dates=['date'])

n_filas = tabla.shape[0]
n_columnas = tabla.shape[1]

print("Filas:", n_filas, "| columnas:", n_columnas)
tabla.head()

Comprobamos que has cargado el fichero correcto:

In [ ]:
assert n_filas == 4097, "Deberías tener 4097 filas: ¿has cargado flotacion_semisintetica_horaria.csv?"
assert n_columnas == 13, "Deberías tener 13 columnas"
print("Correcto")

### 1b. Qué entra en el agrupamiento

k-means solo ve las **9 variables de proceso**: describen cómo estaba operando la planta en cada hora.
`silice` y `fuera_spec` quedan fuera. Si entraran, los grupos se formarían a partir de la calidad y ya
no podríamos preguntarnos si la forma de operar tiene que ver con ella. Las usaremos solo **después**,
para describir cada grupo.

In [ ]:
variables_proceso = ['iron_feed', 'silica_feed', 'almidon_medio', 'amina_media',
                     'caudal_pulpa', 'ph_medio', 'densidad', 'aire_medio', 'nivel_medio']

fraccion_fuera_global = tabla['fuera_spec'].mean()
print(len(variables_proceso), "variables de proceso:", variables_proceso)
print(f"Horas fuera de especificación en toda la tabla: {fraccion_fuera_global:.1%}")

### 1c. Estandarizar

Como en el satélite, k-means mide distancias, así que estandarizamos las 9 variables. Aquí no hay
partición en train y test: no entrenamos un modelo para predecir horas futuras, sino que **describimos**
los seis meses de operación. Por eso el escalador se ajusta con todas las horas.

| Variable | Qué contiene |
|---|---|
| `escalador` | `StandardScaler` ajustado con las 9 variables de todas las horas |
| `X_proceso` | las 9 variables estandarizadas (una fila por hora) |

In [ ]:
escalador = StandardScaler()
X_proceso = escalador.fit_transform(tabla[variables_proceso])
print("X_proceso:", X_proceso.shape[0], "horas x", X_proceso.shape[1], "variables")

**Lo que conviene recordar**

- Una de cada cuatro horas está fuera de especificación (25 %). Es la **referencia** con la que
  compararemos cada grupo.
- Hay un hueco de 319 horas sin registros al principio. En esta tarea no hay que hacer nada con él.

## 2. Agrupar las horas con k-means

En esta tarea fijamos **5 grupos**. A cada grupo lo llamamos **zona de operación**: un conjunto de
horas en las que la planta operaba de forma parecida. En la sección 4 comprobaremos si las horas se
dejan agrupar así.

| Variable | Qué contiene |
|---|---|
| `k_zonas` | número de grupos (5) |
| `modelo_kmeans` | el k-means |
| `zona` | zona asignada a cada hora (de 0 a 4); se guarda también como columna `zona` de `tabla` |
| `horas_por_zona` | número de horas de cada zona |

In [ ]:
k_zonas = 5
modelo_kmeans = KMeans(n_clusters=k_zonas, n_init=10, random_state=0)

# === TODO ===
# Ajusta k-means con X_proceso y obtén la zona de cada hora (una línea).
# pista: zona = modelo_kmeans.fit_predict(X_proceso)
zona = None   # TODO

tabla['zona'] = zona
horas_por_zona = tabla['zona'].value_counts()
horas_por_zona = horas_por_zona.sort_index()
print(horas_por_zona)

**Comprueba:**

In [ ]:
assert zona is not None, "Completa el TODO: zona = modelo_kmeans.fit_predict(X_proceso)"
assert len(horas_por_zona) == 5, "Deberías tener 5 zonas"
assert horas_por_zona.min() == 475 and horas_por_zona.max() == 1165, "La zona más pequeña debería tener 475 horas y la mayor 1165"
print("Correcto")

## 3. Perfil de cada zona: qué calidad da y cómo opera

Ahora sí miramos la calidad. Para cada zona calculamos qué fracción de sus horas salió fuera de
especificación y la sílice media. Si la forma de operar no tuviera nada que ver con la calidad, todas
las zonas estarían cerca del 25 % global.

| Variable | Qué contiene |
|---|---|
| `grupos_zona` | la tabla agrupada por zona |
| `fraccion_fuera_por_zona` | fracción de horas fuera de especificación de cada zona |
| `silice_media_por_zona` | sílice media de cada zona |
| `perfil_calidad` | tabla con las horas, la fracción fuera de especificación y la sílice media de cada zona |
| `comparacion_proceso` | media de cada variable de proceso en cada zona y en toda la tabla |

In [ ]:
grupos_zona = tabla.groupby('zona')

# === TODO ===
# Calcula, para cada zona, la fracción de horas fuera de especificación y la sílice media.
# pista: grupos_zona['fuera_spec'].mean()   y   grupos_zona['silice'].mean()
fraccion_fuera_por_zona = None   # TODO
silice_media_por_zona = None     # TODO

perfil_calidad = pd.DataFrame({
    'horas': horas_por_zona,
    'fraccion_fuera_spec': fraccion_fuera_por_zona,
    'silice_media': silice_media_por_zona,
})
print(f"Referencia global: {fraccion_fuera_global:.1%} de horas fuera de especificación")
perfil_calidad.round(3)

**Comprueba:**

In [ ]:
assert fraccion_fuera_por_zona is not None and silice_media_por_zona is not None, "Completa los dos TODO"
assert abs(fraccion_fuera_por_zona.max() - 0.697) < 0.005, "La zona con más horas fuera de especificación debería tener alrededor del 69,7 %"
assert abs(silice_media_por_zona.max() - 3.70) < 0.01, "La sílice media más alta debería ser de alrededor del 3,70 %"
print("Correcto")

Cómo opera cada zona: la media de cada variable de proceso en cada zona, junto a la media de toda la
tabla como referencia. Ejecuta sin cambiar nada:

In [ ]:
medias_por_zona = grupos_zona[variables_proceso].mean()
comparacion_proceso = medias_por_zona.T
comparacion_proceso['toda la tabla'] = tabla[variables_proceso].mean()
comparacion_proceso.round(1)

**Pregunta 3.1.** ¿Hay alguna zona con una fracción de horas fuera de especificación muy distinta del
25 % global? ¿Cuál es, cuántas horas tiene y cuántas veces supera la referencia global?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.2.** Describe esa zona como se la contarías al operador: ¿en qué variables de proceso se
diferencia del resto? ¿Puedes concluir que esas variables causan la mala calidad? ¿Le serviría saber en
qué zona está la planta para saber si el concentrado de esta hora va a salir mal?

**Tu respuesta:** *escribe aquí*

## 4. ¿Son nítidas las zonas?

En la sección 2 fijamos 5 zonas sin comprobar si las horas se dejan agrupar así. Lo comprobamos con
las herramientas del satélite: el **codo** y la **silueta** para varios *k*, y el **gráfico de
silueta** de las 5 zonas.

Referencia orientativa para la silueta media: por debajo de 0,25, sin estructura clara; de 0,25 a 0,5,
estructura débil; de 0,5 a 0,7, razonable; por encima de 0,7, fuerte.

| Variable | Qué contiene |
|---|---|
| `valores_k` | números de grupos que probamos, de 2 a 8 |
| `inercias`, `siluetas` | inercia y silueta media de cada *k* |
| `silueta_por_hora` | silueta de cada hora con las 5 zonas de la sección 2 |
| `resumen_silueta` | horas, silueta media y horas con silueta negativa de cada zona |

### 4a. Codo y silueta para varios *k*

Repetimos k-means con *k* de 2 a 8 y guardamos la inercia y la silueta media de cada uno. Tarda unos
segundos.

In [ ]:
valores_k = range(2, 9)
inercias = []
siluetas = []

for k in valores_k:
    modelo_k = KMeans(n_clusters=k, n_init=10, random_state=0)
    zona_k = modelo_k.fit_predict(X_proceso)
    inercia_k = modelo_k.inertia_

    # === TODO ===
    # Calcula la silueta media de este agrupamiento (una línea).
    # pista: silhouette_score(X_proceso, zona_k)
    silueta_k = None   # TODO

    inercias.append(inercia_k)
    siluetas.append(silueta_k)
    print(f"k = {k}: inercia = {inercia_k:7.0f}   silueta media = {silueta_k:.3f}")

**Comprueba:**

In [ ]:
assert siluetas[0] is not None, "Completa el TODO: silueta_k = silhouette_score(X_proceso, zona_k)"
posicion_mejor = np.argmax(siluetas)
k_mejor_silueta = valores_k[posicion_mejor]
assert k_mejor_silueta == 5, "La silueta media más alta debería estar en k = 5: revisa silhouette_score(X_proceso, zona_k)"
print("Correcto")

Dibujamos las dos curvas. Ejecuta sin cambiar nada:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(valores_k, inercias, 'o-')
ax[0].set_xlabel('número de grupos (k)')
ax[0].set_ylabel('inercia')
ax[0].set_title('Método del codo')
ax[1].plot(valores_k, siluetas, 'o-', color='C1')
ax[1].axhline(0.25, color='grey', ls=':', label='0,25: por debajo, sin estructura clara')
ax[1].set_ylim(0, 0.3)
ax[1].set_xlabel('número de grupos (k)')
ax[1].set_ylabel('silueta media')
ax[1].set_title('Silueta')
ax[1].legend(loc='lower right')
for eje in ax:
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4b. Gráfico de silueta de las 5 zonas

La silueta media resume todo el agrupamiento en un número. Como en el satélite, el **gráfico de
silueta** dibuja la silueta de **cada hora** como una barra horizontal, agrupada por zona y ordenada de
menor a mayor; la línea roja marca la media de todas las horas. Una hora con silueta negativa está más
cerca de otra zona que de la suya.

Primero, un resumen por zona; después, el gráfico. Ejecuta las dos celdas sin cambiar nada.

In [ ]:
valores_silueta = silhouette_samples(X_proceso, zona)
silueta_por_hora = pd.Series(valores_silueta, index=tabla.index)
silueta_media_por_zona = silueta_por_hora.groupby(tabla['zona']).mean()
es_negativa = silueta_por_hora < 0
negativas_por_zona = es_negativa.groupby(tabla['zona']).sum()
fraccion_silueta_negativa = es_negativa.mean()

resumen_silueta = pd.DataFrame({
    'horas': horas_por_zona,
    'silueta_media': silueta_media_por_zona,
    'horas_silueta_negativa': negativas_por_zona,
})
print(f"Silueta media de todas las horas: {silueta_por_hora.mean():.3f}")
print(f"Horas con silueta negativa: {es_negativa.sum()} ({fraccion_silueta_negativa:.1%})")
resumen_silueta.round(3)

In [ ]:
fig, eje = plt.subplots(figsize=(7, 6))
posicion_inferior = 0

for numero_zona in range(k_zonas):
    es_de_la_zona = tabla['zona'] == numero_zona
    valores_zona = silueta_por_hora[es_de_la_zona]
    valores_zona = np.sort(valores_zona)
    n_zona = len(valores_zona)
    posiciones = np.arange(posicion_inferior, posicion_inferior + n_zona)
    color_zona = f'C{numero_zona}'
    eje.fill_betweenx(posiciones, 0, valores_zona, color=color_zona, alpha=0.8)
    eje.text(0.58, posicion_inferior + n_zona / 2, f'zona {numero_zona}', va='center')
    posicion_inferior = posicion_inferior + n_zona + 40

silueta_media_total = silueta_por_hora.mean()
eje.axvline(silueta_media_total, color='red', ls='--')
eje.axvline(0, color='black', lw=0.8)
eje.set_xlim(-0.3, 0.7)
eje.set_yticks([])
eje.set_xlabel('silueta de cada hora, s(i)')
eje.set_ylabel('horas, agrupadas por zona')
eje.set_title(f'Gráfico de silueta (k = 5); línea roja: media = {silueta_media_total:.2f}')
plt.tight_layout()
plt.show()

**Pregunta 4.1.** ¿Forman las horas grupos nítidos? Apóyate en el codo, en la silueta media y en la
referencia. En el gráfico de silueta, ¿qué zona está peor definida y cuántas de sus horas tienen
silueta negativa? ¿Es la zona de la pregunta 3.1? ¿Le resta eso utilidad para avisar al operador?

**Tu respuesta:** *escribe aquí*

## 5. Ver las zonas en un plano (PCA)

Las horas tienen 9 variables, así que no podemos dibujarlas. Como con Iris en el satélite, la **PCA**
las resume en dos componentes (PC1 y PC2) que sí podemos dibujar. La PCA tampoco ve la calidad: solo
usa `X_proceso`.

| Variable | Qué contiene |
|---|---|
| `pca` | la PCA con dos componentes |
| `componentes` | PC1 y PC2 de cada hora (una fila por hora) |
| `varianza_explicada` | fracción de la varianza que recoge cada componente |
| `varianza_plano` | fracción de la varianza que recogen PC1 y PC2 juntas |
| `cargas` | peso de cada variable de proceso en PC1 y PC2 |

In [ ]:
pca = PCA(n_components=2)

# === TODO ===
# Ajusta la PCA con X_proceso y obtén PC1 y PC2 de cada hora (una línea).
# pista: componentes = pca.fit_transform(X_proceso)
componentes = None   # TODO

varianza_explicada = pca.explained_variance_ratio_
varianza_plano = varianza_explicada.sum()
print(f"PC1 recoge el {varianza_explicada[0]:.1%} de la varianza y PC2 el {varianza_explicada[1]:.1%}")
print(f"Entre las dos: {varianza_plano:.1%}")

**Comprueba:**

In [ ]:
assert componentes is not None, "Completa el TODO: componentes = pca.fit_transform(X_proceso)"
assert componentes.shape == (4097, 2), "Deberías tener 4097 horas x 2 componentes"
assert abs(varianza_plano - 0.493) < 0.005, "PC1 y PC2 deberían recoger alrededor del 49,3 % de la varianza"
print("Correcto")

Las **cargas** dicen cuánto pesa cada variable en cada componente: cuanto mayor es su valor absoluto,
más pesa; el signo indica hacia qué lado del eje empuja. Ejecuta sin cambiar nada:

In [ ]:
cargas = pd.DataFrame(pca.components_.T, index=variables_proceso, columns=['PC1', 'PC2'])
cargas.round(2)

Dibujamos cada hora en el plano PC1–PC2: a la izquierda, coloreada por zona (con los colores del
gráfico de silueta); a la derecha, por calidad. Ejecuta sin cambiar nada:

In [ ]:
pc1 = componentes[:, 0]
pc2 = componentes[:, 1]
es_fuera = tabla['fuera_spec'] == 1
es_dentro = tabla['fuera_spec'] == 0

fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for numero_zona in range(k_zonas):
    es_de_la_zona = tabla['zona'] == numero_zona
    ax[0].scatter(pc1[es_de_la_zona], pc2[es_de_la_zona], s=4, alpha=0.5,
                  color=f'C{numero_zona}', label=f'zona {numero_zona}')
ax[0].set_title('Horas por zona')
ax[0].legend(markerscale=4)

ax[1].scatter(pc1[es_dentro], pc2[es_dentro], s=4, alpha=0.3, color='grey', label='en especificación')
ax[1].scatter(pc1[es_fuera], pc2[es_fuera], s=4, alpha=0.5, color='red', label='fuera de especificación')
ax[1].set_title('Horas por calidad')
ax[1].legend(markerscale=4)

for eje in ax:
    eje.set_xlabel('PC1')
    eje.grid(alpha=0.3)
ax[0].set_ylabel('PC2')
plt.tight_layout()
plt.show()

**Pregunta 5.1.** ¿Qué porcentaje de la varianza recoge el plano PC1–PC2? Según las cargas, ¿qué
variables pesan más en PC1 y cuáles en PC2? ¿Dónde queda la zona de la pregunta 3.1: es una isla
separada del resto o un extremo de la nube? ¿Encaja con lo que viste en la sección 4?

**Tu respuesta:** *escribe aquí*

> **Antes de entregar:** reinicia el kernel y ejecuta todo (en VS Code, «Restart» y luego «Run All»; en
> Colab, «Entorno de ejecución → Reiniciar y ejecutar todo»). Comprueba que no hay errores y que todas
> las celdas «Comprueba» dicen «Correcto». Guarda y sube el `.ipynb` al Campus Virtual.